# Sesión 5 Extra: Fine-tuning de modelos de lenguaje

Este notebook complementa la sesión de evaluación y robustez. Hasta ahora hemos trabajado con prompting, RAG y agentes. Falta una pieza importante: fine-tuning.

El fine-tuning no es una alternativa universal ni una forma fiable de “meter documentación” en un modelo. Es una técnica para modificar el comportamiento del modelo a partir de ejemplos. Puede ser muy útil, pero solo cuando el problema realmente lo necesita y cuando tenemos datos de calidad.

El objetivo de este notebook es aprender a decidir cuándo tiene sentido, cómo preparar datos, cómo evaluar antes y después, y qué cambia entre fine-tuning de proveedor y fine-tuning open source con LoRA/QLoRA.

## Objetivos

Al terminar deberías entender:

- Diferencia entre prompting, RAG, agentes y fine-tuning.
- Qué problemas resuelve bien el fine-tuning y cuáles no.
- Cómo preparar un dataset JSONL de entrenamiento y validación.
- Cómo validar estructura, roles, tamaños y distribución de ejemplos.
- Cómo plantear un baseline antes de entrenar.
- Cómo lanzar de forma segura un fine-tuning de proveedor sin ejecutarlo accidentalmente.
- Qué son LoRA y QLoRA en modelos open source.
- Qué limitaciones prácticas tienen hardware, licencias, datos y despliegue.

## 0. Preparación del entorno

Este notebook tiene dos partes:

1. Una parte ejecutable en local/API: validación de datasets, baseline y preparación de fine-tuning de proveedor.
2. Una parte opcional de LoRA/QLoRA: pensada para ejecutarse solo si tienes GPU y quieres profundizar en modelos open source.

Las celdas que pueden generar coste o requerir GPU están comentadas o encapsuladas para evitar ejecución accidental.

In [ ]:
# Si estás en un entorno limpio, descomenta esta celda.
# %pip install -q -U openai python-dotenv pydantic pandas tiktoken

# Para la sección opcional open source con GPU:
# %pip install -q -U transformers datasets peft accelerate bitsandbytes trl torch

In [ ]:
import getpass
import json
import os
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError

WORK_DIR = Path.cwd()
if (WORK_DIR / "sesion_05").exists():
    SESSION_DIR = WORK_DIR / "sesion_05"
else:
    SESSION_DIR = WORK_DIR

DATA_DIR = SESSION_DIR / "data"
TRAIN_PATH = DATA_DIR / "finetune_style_train.jsonl"
VALIDATION_PATH = DATA_DIR / "finetune_style_validation.jsonl"

load_dotenv(SESSION_DIR / ".env")

BASE_MODEL = "gpt-4.1-mini"
print("Train:", TRAIN_PATH)
print("Validation:", VALIDATION_PATH)

## 1. Fine-tuning como decisión arquitectónica

Antes de entrenar nada, hay que decidir si fine-tuning es la herramienta correcta.

**Prompting** es adecuado cuando el comportamiento deseado se puede explicar con instrucciones razonablemente cortas.

**RAG** es adecuado cuando el problema depende de conocimiento externo, cambiante, auditable o con permisos.

**Agentes** son adecuados cuando el sistema debe decidir entre herramientas o pasos.

**Fine-tuning** es adecuado cuando queremos que el modelo aprenda un patrón repetitivo a partir de ejemplos: formato, estilo, clasificación, extracción, tono, taxonomía o forma de razonar dentro de un dominio estrecho.

La pregunta no es “¿puedo hacer fine-tuning?”, sino “¿qué comportamiento observable quiero mejorar y cómo sabré que ha mejorado?”.

### 1.1. Cuándo sí y cuándo no

Fine-tuning suele tener sentido para:

- Respuestas con estilo muy específico.
- Clasificación de tickets, emails o documentos con taxonomía estable.
- Extracción estructurada repetitiva.
- Conversión de lenguaje natural a formato interno.
- Reducción de prompts largos cuando hay muchos ejemplos consistentes.
- Adaptación a tono corporativo.

No suele ser la primera opción para:

- Documentación que cambia cada semana.
- Preguntas donde hay que citar fuentes.
- Acceso a datos con permisos por usuario.
- Incorporar hechos que deben actualizarse y auditarse.
- Corregir un mal dataset o una mala definición del producto.

Para conocimiento interno, normalmente empezamos por RAG. Para comportamiento repetitivo, consideramos fine-tuning.

## 2. Dataset de fine-tuning

Vamos a trabajar con un dataset pequeño de estilo/formato. No pretende entrenar un modelo excelente; pretende mostrar el flujo correcto.

Cada línea JSONL contiene una conversación con `messages`. El modelo aprende a producir respuestas con el estilo de los ejemplos: breves, claras, profesionales y con recomendaciones operativas.

In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                rows.append(json.loads(line))
    return rows


train_rows = read_jsonl(TRAIN_PATH)
validation_rows = read_jsonl(VALIDATION_PATH)
print("Train examples:", len(train_rows))
print("Validation examples:", len(validation_rows))
print(json.dumps(train_rows[0], ensure_ascii=False, indent=2))

In [ ]:
class ChatMessage(BaseModel):
    role: str
    content: str


class FineTuneExample(BaseModel):
    messages: list[ChatMessage] = Field(min_length=2)


def validate_examples(rows: list[dict[str, Any]], *, name: str) -> pd.DataFrame:
    records = []
    for idx, row in enumerate(rows):
        try:
            example = FineTuneExample.model_validate(row)
            roles = [message.role for message in example.messages]
            assistant_count = roles.count("assistant")
            user_count = roles.count("user")
            valid = assistant_count >= 1 and user_count >= 1 and all(message.content.strip() for message in example.messages)
            error = None if valid else "Faltan roles user/assistant o hay contenido vacío"
        except ValidationError as exc:
            roles = []
            valid = False
            error = str(exc)
        records.append({"dataset": name, "idx": idx, "valid": valid, "roles": roles, "error": error})
    return pd.DataFrame(records)


validation_report = pd.concat(
    [validate_examples(train_rows, name="train"), validate_examples(validation_rows, name="validation")],
    ignore_index=True,
)
validation_report

### 2.1. Validaciones adicionales

Además de la estructura, conviene revisar:

- Longitud de ejemplos.
- Distribución de roles.
- Duplicados exactos.
- Presencia de datos sensibles.
- Consistencia del estilo deseado.
- Separación real entre entrenamiento y validación.

Un modelo fine-tuned hereda los problemas de los datos. Si los ejemplos son inconsistentes, el modelo aprenderá inconsistencia.

In [ ]:
def example_text(example: dict[str, Any]) -> str:
    return "\n".join(f"{m['role']}: {m['content']}" for m in example["messages"])


def dataset_profile(rows: list[dict[str, Any]]) -> pd.DataFrame:
    records = []
    for idx, row in enumerate(rows):
        text = example_text(row)
        roles = [m["role"] for m in row["messages"]]
        records.append(
            {
                "idx": idx,
                "chars": len(text),
                "messages": len(row["messages"]),
                "roles": ",".join(roles),
                "assistant_chars": sum(len(m["content"]) for m in row["messages"] if m["role"] == "assistant"),
            }
        )
    return pd.DataFrame(records)


profile = dataset_profile(train_rows)
profile.describe(include="all")

In [ ]:
SENSITIVE_TERMS = ["api_key", "password", "contraseña", "token", "secret", "secreto"]


def scan_sensitive(rows: list[dict[str, Any]]) -> pd.DataFrame:
    findings = []
    for idx, row in enumerate(rows):
        text = example_text(row).lower()
        matched = [term for term in SENSITIVE_TERMS if term in text]
        if matched:
            findings.append({"idx": idx, "matched": matched})
    return pd.DataFrame(findings)


scan_sensitive(train_rows)

## 3. Baseline antes de entrenar

Nunca deberíamos hacer fine-tuning sin baseline. Primero medimos cómo responde el modelo base con prompting. Después entrenamos, evaluamos el modelo ajustado y comparamos.

Si no hay una mejora medible, el fine-tuning no compensa coste, complejidad y mantenimiento.

In [ ]:
BASELINE_SYSTEM_PROMPT = """
Eres un asistente de soporte interno. Responde de forma breve, clara y con tono profesional.
No inventes políticas. Si falta información, sugiere consultar al área responsable.
""".strip()

baseline_questions = [
    "No puedo acceder a la VPN",
    "Tengo un gasto de proveedor de 900 euros",
    "¿Puedo enviar datos de cliente a mi correo personal?",
    "Quiero comprar software para mi equipo",
]

pd.DataFrame({"question": baseline_questions})

In [ ]:
# Ejecución opcional del baseline con OpenAI.
# from openai import OpenAI
# if not os.getenv("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
# client = OpenAI()

# def run_baseline(client, question: str) -> str:
#     response = client.responses.create(
#         model=BASE_MODEL,
#         input=[
#             {"role": "system", "content": BASELINE_SYSTEM_PROMPT},
#             {"role": "user", "content": question},
#         ],
#     )
#     return response.output_text

# baseline_outputs = [run_baseline(client, q) for q in baseline_questions]
# pd.DataFrame({"question": baseline_questions, "answer": baseline_outputs})

## 4. Fine-tuning de proveedor: flujo seguro

El flujo general con un proveedor suele ser:

1. Preparar JSONL de entrenamiento y validación.
2. Validar estructura y calidad.
3. Subir archivos.
4. Crear job de fine-tuning.
5. Monitorizar estado.
6. Evaluar modelo resultante contra baseline.
7. Decidir si se usa, se reentrena o se descarta.

Las siguientes celdas muestran el flujo, pero las llamadas que generan coste quedan comentadas explícitamente.

In [ ]:
def estimate_dataset_size(path: Path) -> dict[str, Any]:
    text = path.read_text(encoding="utf-8")
    # Estimación grosera: 1 token ~= 4 caracteres en muchos textos latinos.
    estimated_tokens = len(text) / 4
    return {"path": str(path), "bytes": path.stat().st_size, "estimated_tokens": int(estimated_tokens)}


pd.DataFrame([estimate_dataset_size(TRAIN_PATH), estimate_dataset_size(VALIDATION_PATH)])

In [ ]:
# Subida de archivos opcional. Descomenta solo si quieres crear recursos reales en tu cuenta.
# from openai import OpenAI
# client = OpenAI()

# train_file = client.files.create(file=TRAIN_PATH.open("rb"), purpose="fine-tune")
# validation_file = client.files.create(file=VALIDATION_PATH.open("rb"), purpose="fine-tune")
# print(train_file.id, validation_file.id)

In [ ]:
# Creación opcional del job de fine-tuning. Mantener comentado para evitar coste accidental.
# fine_tune_job = client.fine_tuning.jobs.create(
#     training_file=train_file.id,
#     validation_file=validation_file.id,
#     model=BASE_MODEL,
#     suffix="pontia-soporte-estilo",
# )
# print(fine_tune_job.id)

In [ ]:
# Monitorización opcional.
# job = client.fine_tuning.jobs.retrieve(fine_tune_job.id)
# print(job.status)
# events = client.fine_tuning.jobs.list_events(fine_tune_job.id, limit=10)
# for event in events.data:
#     print(event.created_at, event.message)

### 4.1. Evaluar el modelo fine-tuned

Cuando el job termina, el proveedor devuelve un identificador de modelo ajustado. Ese modelo debe evaluarse con el mismo dataset de evaluación que el baseline.

No basta con probar una pregunta y decir que “suena mejor”. Hay que comparar estilo, completitud, seguridad, coste y latencia.

In [ ]:
# Ejemplo de evaluación opcional del modelo ajustado.
# FINE_TUNED_MODEL = "ft:gpt-4.1-mini:org:project:suffix:id"

# def run_finetuned(client, question: str) -> str:
#     response = client.responses.create(
#         model=FINE_TUNED_MODEL,
#         input=[{"role": "user", "content": question}],
#     )
#     return response.output_text

# ft_outputs = [run_finetuned(client, q) for q in baseline_questions]
# comparison = pd.DataFrame({"question": baseline_questions, "baseline": baseline_outputs, "fine_tuned": ft_outputs})
# comparison

## 5. Rúbrica para decidir si el fine-tuning compensa

Una posible rúbrica:

- **Consistencia de estilo:** ¿responde con el formato esperado sin repetir instrucciones largas?
- **Precisión:** ¿mantiene o mejora la corrección?
- **Seguridad:** ¿no aumenta respuestas indebidas?
- **Coste:** ¿reduce tokens de prompt o compensa el coste de entrenamiento?
- **Latencia:** ¿mejora, empeora o se mantiene?
- **Mantenibilidad:** ¿el comportamiento cambiará a menudo? Si cambia mucho, quizá no conviene fine-tuning.

Fine-tuning no debería adoptarse por elegancia técnica, sino por mejora observable.

In [ ]:
ft_decision_template = pd.DataFrame(
    [
        {"criterion": "style_consistency", "baseline_score_1_5": None, "fine_tuned_score_1_5": None, "notes": ""},
        {"criterion": "accuracy", "baseline_score_1_5": None, "fine_tuned_score_1_5": None, "notes": ""},
        {"criterion": "safety", "baseline_score_1_5": None, "fine_tuned_score_1_5": None, "notes": ""},
        {"criterion": "cost", "baseline_score_1_5": None, "fine_tuned_score_1_5": None, "notes": ""},
        {"criterion": "maintainability", "baseline_score_1_5": None, "fine_tuned_score_1_5": None, "notes": ""},
    ]
)
ft_decision_template

## 6. Fine-tuning open source: LoRA y QLoRA

En modelos open source, no solemos ajustar todos los pesos del modelo base. Es caro y requiere mucho hardware. En su lugar se usan técnicas de parameter-efficient fine-tuning.

**LoRA** añade matrices pequeñas entrenables sobre ciertas capas del modelo. El modelo base queda congelado y entrenamos solo esos adaptadores.

**QLoRA** combina cuantización del modelo base con LoRA para reducir memoria. Permite entrenar adaptadores sobre modelos relativamente grandes con menos GPU, aunque sigue requiriendo cuidado técnico.

Ventajas:

- Más control sobre modelo, datos y despliegue.
- Posibilidad de inferencia local o privada.
- Adaptadores pequeños y versionables.

Costes:

- Complejidad de entorno.
- Necesidad de GPU para entrenamientos útiles.
- Gestión de licencias.
- Evaluación y serving por cuenta propia.

In [ ]:
# Comprobación opcional de GPU.
# import torch
# print("CUDA disponible:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print(torch.cuda.get_device_name(0))

In [ ]:
# Esqueleto opcional de LoRA/QLoRA. No ejecutar sin revisar hardware y dependencias.

# from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# from trl import SFTTrainer, SFTConfig
# import torch

# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
# )

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# tokenizer.pad_token = tokenizer.eos_token

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     quantization_config=quant_config,
#     device_map="auto",
# )
# model = prepare_model_for_kbit_training(model)

# lora_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     lora_dropout=0.05,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
#     task_type="CAUSAL_LM",
# )
# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

### 6.1. Formato de datos para modelos instruct open source

Cada familia de modelos usa una plantilla de chat distinta. En OpenAI enviamos `messages`; en Hugging Face normalmente hay que aplicar `tokenizer.apply_chat_template` para convertir mensajes a texto de entrenamiento.

Este detalle importa mucho. Si entrenas con una plantilla incorrecta, el modelo aprende un formato que luego no coincide con la inferencia.

In [ ]:
def openai_messages_to_plain_text(example: dict[str, Any]) -> str:
    """Representación simple para inspección humana, no plantilla final de entrenamiento."""
    return "\n".join(f"[{m['role'].upper()}] {m['content']}" for m in example["messages"])


print(openai_messages_to_plain_text(train_rows[0]))

In [ ]:
# Ejemplo conceptual de preparación con chat template.
# def format_for_instruct_model(example):
#     return tokenizer.apply_chat_template(
#         example["messages"],
#         tokenize=False,
#         add_generation_prompt=False,
#     )

# dataset = load_dataset("json", data_files={"train": str(TRAIN_PATH), "validation": str(VALIDATION_PATH)})
# dataset = dataset.map(lambda row: {"text": format_for_instruct_model(row)})

### 6.2. Entrenamiento opcional con TRL

El entrenamiento real depende de GPU, memoria, modelo, dataset y versión de librerías. Por eso esta celda queda como plantilla comentada.

La idea general es:

1. Cargar modelo y tokenizer.
2. Preparar cuantización si aplica.
3. Añadir adaptadores LoRA.
4. Convertir datos al formato de chat del modelo.
5. Entrenar con pocas épocas.
6. Guardar adaptador.
7. Evaluar contra baseline.

In [ ]:
# training_args = SFTConfig(
#     output_dir="outputs/lora-soporte-interno",
#     num_train_epochs=1,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,
#     learning_rate=2e-4,
#     logging_steps=5,
#     save_steps=50,
#     max_seq_length=1024,
# )

# trainer = SFTTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset["train"],
#     eval_dataset=dataset["validation"],
#     peft_config=lora_config,
#     dataset_text_field="text",
#     tokenizer=tokenizer,
# )

# trainer.train()
# trainer.save_model("outputs/lora-soporte-interno/final")

## 7. Riesgos específicos de fine-tuning

Fine-tuning añade riesgos propios:

- **Sobreajuste:** el modelo memoriza ejemplos y generaliza mal.
- **Datos sensibles:** puede memorizar o reproducir información que no debía estar en entrenamiento.
- **Regresión de seguridad:** mejorar estilo puede empeorar rechazo de peticiones peligrosas.
- **Falsa sensación de conocimiento:** el modelo puede responder con seguridad aunque el hecho haya cambiado.
- **Coste de mantenimiento:** cada cambio de política puede requerir nuevos datos y reentrenamiento.
- **Evaluación insuficiente:** sin baseline y validación, no sabes si el ajuste ayuda.

Por eso fine-tuning debe ir unido a evaluación, no sustituirla.

## 7.1. Mitos frecuentes sobre fine-tuning

Hay varios malentendidos recurrentes.

**Mito 1: fine-tuning mete conocimiento en el modelo.** En realidad, puede modificar probabilidades de respuesta, pero no es una base de conocimiento auditable. Si necesitas saber de dónde sale una política, RAG sigue siendo mejor.

**Mito 2: fine-tuning arregla cualquier mal prompt.** Si la tarea está mal definida o los ejemplos son contradictorios, el fine-tuning amplifica el problema.

**Mito 3: con pocos ejemplos siempre basta.** Algunos comportamientos de estilo pueden mejorar con pocos ejemplos, pero tareas complejas requieren cobertura real y validación.

**Mito 4: el modelo fine-tuned es automáticamente más barato.** Puede reducir tokens de prompt, pero añade coste de entrenamiento, evaluación y mantenimiento.

**Mito 5: fine-tuning sustituye seguridad.** Un modelo ajustado puede seguir aceptando instrucciones peligrosas si las herramientas y permisos no están bien diseñados.

## 7.2. Estrategia de datos

El dataset es el producto principal de un proyecto de fine-tuning. No es un subproducto.

Una estrategia razonable incluye:

- Definir el comportamiento objetivo en lenguaje natural.
- Crear ejemplos positivos que lo representen.
- Crear ejemplos negativos o de rechazo cuando el asistente deba poner límites.
- Mantener separación entre entrenamiento, validación y test.
- Revisar manualmente consistencia de tono y formato.
- Eliminar datos sensibles, identificadores reales y casos con información no autorizada.
- Versionar el dataset y registrar qué modelo se entrenó con qué versión.

Si el dataset viene de conversaciones reales, conviene anonimizarlo y filtrar ruido. Las conversaciones reales suelen contener interrupciones, datos personales, errores y respuestas malas del asistente anterior. Entrenar directamente con eso puede degradar el modelo.

## 7.3. Fine-tuning y RAG juntos

Fine-tuning y RAG no compiten necesariamente. En muchos sistemas serios se combinan.

Por ejemplo:

- Fine-tuning para que el modelo responda siempre con un formato corporativo compacto.
- RAG para aportar políticas actualizadas y fuentes.
- Herramientas para consultar datos transaccionales.
- Guardrails para controlar seguridad y permisos.

En ese diseño, el fine-tuning enseña “cómo responder”, mientras que RAG aporta “con qué información responder”. Separar esas responsabilidades evita intentar resolver con entrenamiento un problema que pertenece a recuperación documental.

## 7.4. Despliegue de modelos ajustados

Entrenar es solo una parte. Después hay que operar el modelo.

En proveedor, el despliegue suele consistir en usar un identificador de modelo fine-tuned. Aun así, debes gestionar permisos, coste, latencia, fallback y evaluación continua.

En open source, además necesitas decidir:

- Dónde se servirá el modelo.
- Si se cargará modelo completo o adaptador LoRA sobre base.
- Qué cuantización se usará en inferencia.
- Cómo se versionan adaptadores y tokenizer.
- Qué GPU o runtime se necesita.
- Cómo se escalan réplicas si hay tráfico.

Este coste operativo es una de las razones por las que fine-tuning open source debe reservarse para casos donde el control adicional compensa la complejidad.

## 7.5. Checklist antes de entrenar

Antes de lanzar un fine-tuning, revisa:

- Hay un baseline medido.
- Existe una métrica o rúbrica de mejora.
- El dataset tiene ejemplos suficientes y consistentes.
- No hay secretos ni datos sensibles.
- La validación está separada del entrenamiento.
- El comportamiento deseado no se resuelve mejor con RAG o prompting.
- Se ha estimado coste de entrenamiento e inferencia.
- Hay plan de rollback si el modelo ajustado empeora.
- Hay casos de seguridad en evaluación.
- Se documenta versión de dataset, modelo base y parámetros.

Si varios puntos no se cumplen, probablemente todavía no toca entrenar.

## 7.6. Selección de modelo base

La elección del modelo base condiciona todo el proyecto. Un modelo pequeño puede ser barato y rápido, pero quizá no tenga capacidad suficiente para seguir instrucciones complejas. Un modelo grande puede necesitar menos fine-tuning, pero ser más caro de operar.

Para proveedor, la decisión suele depender de disponibilidad de fine-tuning, coste por token, latencia y calidad de baseline. Para open source, además entran tamaño, licencia, idioma, plantilla de chat, soporte de cuantización y ecosistema de serving.

Una regla práctica: no ajustes un modelo que ya falla claramente en el baseline por falta de capacidad general. Fine-tuning puede orientar comportamiento, pero no convierte un modelo débil en un modelo frontier. Primero elige un modelo base razonable; después decide si merece la pena adaptarlo.

## 7.7. Evaluación de comportamiento frente a evaluación de conocimiento

En fine-tuning conviene separar dos preguntas.

La primera es: ¿el modelo se comporta como queremos? Esto incluye tono, formato, brevedad, estructura, rechazo de solicitudes y consistencia. Aquí fine-tuning puede aportar mucho.

La segunda es: ¿el modelo sabe hechos correctos y actualizados? Para esta pregunta, fine-tuning suele ser peor herramienta que RAG o herramientas de consulta. Incluso si el modelo aprende que “los gastos superiores a 500 euros requieren aprobación”, esa política puede cambiar. Además, no podremos citar una fuente documental con la misma claridad.

Por eso el dataset de fine-tuning debería centrarse en comportamiento. Si contiene hechos, deben ser estables, no sensibles y estar alineados con el uso esperado.

## 7.8. Preparación de datos conversacionales reales

Si partes de conversaciones reales de soporte, no las uses sin procesar. Primero hay que convertirlas en ejemplos de alta calidad.

Proceso recomendado:

1. Seleccionar conversaciones representativas.
2. Eliminar datos personales, identificadores, emails, teléfonos y secretos.
3. Corregir respuestas malas del asistente o del operador humano.
4. Unificar tono y formato.
5. Reducir conversaciones largas a turnos útiles.
6. Añadir ejemplos de rechazo y escalado.
7. Separar ejemplos por dominio para evitar mezclar políticas incompatibles.

El fine-tuning aprende de lo que le das. Si entrenas con conversaciones desordenadas, respuestas contradictorias o datos sensibles, el problema no será el algoritmo; será el dataset.

## 7.9. Parámetros de entrenamiento en LoRA

En LoRA/QLoRA, algunos parámetros importan especialmente:

- `r`: rango de los adaptadores. Valores mayores dan más capacidad, pero más memoria y riesgo de sobreajuste.
- `lora_alpha`: escala de la actualización LoRA.
- `lora_dropout`: regularización; útil cuando el dataset es pequeño.
- `target_modules`: capas donde se insertan adaptadores. En transformers suelen ser proyecciones de atención.
- `learning_rate`: demasiado alto puede destruir comportamiento; demasiado bajo puede no aprender.
- `max_seq_length`: debe cubrir tus ejemplos reales sin desperdiciar memoria.
- `gradient_accumulation_steps`: permite simular batches mayores con poca memoria.

No existe una combinación universal. Por eso incluso en open source la evaluación sigue siendo central.

## 7.10. Cuándo parar

Un proyecto de fine-tuning puede convertirse en una búsqueda infinita de pequeñas mejoras. Define condiciones de parada antes de empezar.

Ejemplos:

- La mejora frente a baseline no supera el umbral acordado.
- El modelo ajustado empeora seguridad o rechazos.
- El coste operativo supera el ahorro en tokens.
- Los errores restantes se deben a falta de conocimiento documental, no a comportamiento.
- El dataset disponible ya no representa casos reales.

Saber parar es parte de la ingeniería. A veces la mejor conclusión de un experimento de fine-tuning es no usar fine-tuning.

## 8. Recapitulación

En este notebook hemos situado fine-tuning dentro del mapa de decisiones:

- Prompting para instrucciones simples.
- RAG para conocimiento externo y auditable.
- Agentes para decisión dinámica y herramientas.
- Fine-tuning para comportamiento repetitivo aprendido de ejemplos.

También hemos preparado datasets JSONL, validaciones, baseline, flujo seguro de proveedor y una introducción práctica a LoRA/QLoRA open source.

La conclusión práctica es: fine-tuning es valioso cuando mejora una métrica concreta sobre un baseline. Si no puedes medir esa mejora, todavía no estás listo para entrenar.

## 9. Ejercicios propuestos

1. Añade diez ejemplos nuevos al dataset de entrenamiento manteniendo el mismo estilo.
2. Crea una rúbrica de evaluación de estilo y puntúa baseline frente a ejemplos esperados.
3. Añade ejemplos negativos donde el asistente deba rechazar una petición insegura.
4. Separa un conjunto de test que no se use ni en entrenamiento ni en validación.
5. Investiga qué modelo open source pequeño podrías ajustar en tu hardware y documenta requisitos.
6. Diseña un experimento para comparar prompt largo frente a fine-tuning con prompt corto.